In [1]:
"""
DATA PREPARATION NOTEBOOK
-------------------------

Goal:
Prepare the 20 Newsgroups dataset for semantic search and clustering.

Dataset characteristics:
- ~20,000 documents
- 20 topic categories
- Raw format: folder per category, each file = one document

Problems with raw dataset:
- Email headers
- Metadata (From, Subject, Organization)
- Quoted replies
- Email addresses
- Special characters

Our preprocessing decisions:
1. Remove email headers (metadata not useful for semantic meaning)
2. Remove quoted replies (avoid duplicated content)
3. Remove email addresses
4. Normalize text
5. Remove excessive punctuation

We intentionally DO NOT:
- Remove stopwords aggressively
- Perform stemming/lemmatization

Reason:
Embedding models already handle linguistic variations well, and overly
aggressive preprocessing can damage semantic meaning.
"""

'\nDATA PREPARATION NOTEBOOK\n-------------------------\n\nGoal:\nPrepare the 20 Newsgroups dataset for semantic search and clustering.\n\nDataset characteristics:\n- ~20,000 documents\n- 20 topic categories\n- Raw format: folder per category, each file = one document\n\nProblems with raw dataset:\n- Email headers\n- Metadata (From, Subject, Organization)\n- Quoted replies\n- Email addresses\n- Special characters\n\nOur preprocessing decisions:\n1. Remove email headers (metadata not useful for semantic meaning)\n2. Remove quoted replies (avoid duplicated content)\n3. Remove email addresses\n4. Normalize text\n5. Remove excessive punctuation\n\nWe intentionally DO NOT:\n- Remove stopwords aggressively\n- Perform stemming/lemmatization\n\nReason:\nEmbedding models already handle linguistic variations well, and overly\naggressive preprocessing can damage semantic meaning.\n'

In [2]:
import os
import re
import pandas as pd
from tqdm import tqdm

In [3]:
DATASET_PATH = "../data/raw/20_newsgroups"

print("Categories found:")
print(os.listdir(DATASET_PATH))

Categories found:
['alt.atheism', 'comp.graphics', 'comp.os.ms-windows.misc', 'comp.sys.ibm.pc.hardware', 'comp.sys.mac.hardware', 'comp.windows.x', 'misc.forsale', 'rec.autos', 'rec.motorcycles', 'rec.sport.baseball', 'rec.sport.hockey', 'sci.crypt', 'sci.electronics', 'sci.med', 'sci.space', 'soc.religion.christian', 'talk.politics.guns', 'talk.politics.mideast', 'talk.politics.misc', 'talk.religion.misc']


In [4]:
def remove_headers(text):
    """
    Remove email-style headers from the document.

    Headers usually appear before the first blank line.
    Example headers:
    - From:
    - Subject:
    - Organization:
    - Lines:
    """
    parts = text.split("\n\n", 1)

    if len(parts) > 1:
        return parts[1]

    return text


def remove_quotes(text):
    """
    Remove quoted replies.

    Quoted lines start with '>'.
    These usually represent earlier messages in the thread and
    introduce redundant information.
    """
    lines = text.split("\n")
    cleaned_lines = [line for line in lines if not line.strip().startswith(">")]

    return "\n".join(cleaned_lines)


def basic_clean(text):
    """
    Perform light text normalization.

    Steps:
    - lowercase text
    - remove email addresses
    - remove non-alphabetic characters
    - collapse multiple spaces
    """

    text = text.lower()

    # remove email addresses
    text = re.sub(r'\S+@\S+', ' ', text)

    # remove special characters
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)

    # remove extra spaces
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

In [5]:
texts = []
categories = []
doc_ids = []

doc_counter = 0

for category in tqdm(os.listdir(DATASET_PATH)):

    category_path = os.path.join(DATASET_PATH, category)

    if not os.path.isdir(category_path):
        continue

    for filename in os.listdir(category_path):

        file_path = os.path.join(category_path, filename)

        try:
            with open(file_path, "r", encoding="latin1") as f:

                raw_text = f.read()

                texts.append(raw_text)
                categories.append(category)
                doc_ids.append(doc_counter)

                doc_counter += 1

        except Exception:
            continue

100%|██████████| 20/20 [03:44<00:00, 11.22s/it]


In [6]:
df = pd.DataFrame({
    "doc_id": doc_ids,
    "category": categories,
    "raw_text": texts
})

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (19997, 3)


,doc_id,category,raw_text
0,0,alt.atheism,Xref: cantaloupe.srv.cs.cmu.edu alt.atheism:49...
1,1,alt.atheism,Xref: cantaloupe.srv.cs.cmu.edu alt.atheism:51...
2,2,alt.atheism,Newsgroups: alt.atheism\nPath: cantaloupe.srv....
3,3,alt.atheism,Xref: cantaloupe.srv.cs.cmu.edu alt.atheism:51...
4,4,alt.atheism,Xref: cantaloupe.srv.cs.cmu.edu alt.atheism:51...


In [7]:
df["category"].value_counts()

category
alt.atheism                 1000
comp.graphics               1000
comp.os.ms-windows.misc     1000
comp.sys.ibm.pc.hardware    1000
comp.sys.mac.hardware       1000
comp.windows.x              1000
misc.forsale                1000
rec.autos                   1000
rec.motorcycles             1000
rec.sport.baseball          1000
rec.sport.hockey            1000
sci.crypt                   1000
sci.electronics             1000
sci.med                     1000
sci.space                   1000
talk.politics.guns          1000
talk.politics.mideast       1000
talk.politics.misc          1000
talk.religion.misc          1000
soc.religion.christian       997
Name: count, dtype: int64

In [8]:
print(df["raw_text"][0][:800])

Xref: cantaloupe.srv.cs.cmu.edu alt.atheism:49960 alt.atheism.moderated:713 news.answers:7054 alt.answers:126
Path: cantaloupe.srv.cs.cmu.edu!crabapple.srv.cs.cmu.edu!bb3.andrew.cmu.edu!news.sei.cmu.edu!cis.ohio-state.edu!magnus.acs.ohio-state.edu!usenet.ins.cwru.edu!agate!spool.mu.edu!uunet!pipex!ibmpcug!mantis!mathew
From: mathew <mathew@mantis.co.uk>
Newsgroups: alt.atheism,alt.atheism.moderated,news.answers,alt.answers
Subject: Alt.Atheism FAQ: Atheist Resources
Summary: Books, addresses, music -- anything related to atheism
Keywords: FAQ, atheism, books, music, fiction, addresses, contacts
Message-ID: <19930329115719@mantis.co.uk>
Date: Mon, 29 Mar 1993 11:57:19 GMT
Expires: Thu, 29 Apr 1993 11:57:19 GMT
Followup-To: alt.atheism
Distribution: world
Organization: Mantis Consultants, Ca


In [9]:
df["text_no_headers"] = df["raw_text"].apply(remove_headers)

df["text_no_quotes"] = df["text_no_headers"].apply(remove_quotes)

df["clean_text"] = df["text_no_quotes"].apply(basic_clean)

In [10]:
print("Original text snippet:\n")
print(df["raw_text"][0][:400])

print("\nCleaned text snippet:\n")
print(df["clean_text"][0][:400])

Original text snippet:

Xref: cantaloupe.srv.cs.cmu.edu alt.atheism:49960 alt.atheism.moderated:713 news.answers:7054 alt.answers:126
Path: cantaloupe.srv.cs.cmu.edu!crabapple.srv.cs.cmu.edu!bb3.andrew.cmu.edu!news.sei.cmu.edu!cis.ohio-state.edu!magnus.acs.ohio-state.edu!usenet.ins.cwru.edu!agate!spool.mu.edu!uunet!pipex!ibmpcug!mantis!mathew
From: mathew <mathew@mantis.co.uk>
Newsgroups: alt.atheism,alt.atheism.moderate

Cleaned text snippet:

archive name atheism resources alt atheism archive name resources last modified december version atheist resources addresses of atheist organizations usa freedom from religion foundation darwin fish bumper stickers and assorted other atheist paraphernalia are available from the freedom from religion foundation in the us write to ffrf p o box madison wi telephone evolution designs evolution designs


In [11]:
df["text_length"] = df["clean_text"].apply(len)

print("Average text length:", df["text_length"].mean())
print("Minimum text length:", df["text_length"].min())
print("Maximum text length:", df["text_length"].max())

Average text length: 1117.0001000150023
Minimum text length: 0
Maximum text length: 66184


In [12]:
df["text_length"].describe()

count    19997.000000
mean      1117.000100
std       2855.567155
min          0.000000
25%        302.000000
50%        547.000000
75%       1035.000000
max      66184.000000
Name: text_length, dtype: float64

In [13]:
df = df[df["clean_text"].str.strip() != ""]

print("Dataset after removing empty documents:", df.shape)

Dataset after removing empty documents: (19955, 7)


In [14]:
df_final = df[["doc_id", "category", "clean_text"]]

df_final.head()

,doc_id,category,clean_text
0,0,alt.atheism,archive name atheism resources alt atheism arc...
1,1,alt.atheism,archive name atheism introduction alt atheism ...
2,2,alt.atheism,in article charley wingate writes the argument...
3,3,alt.atheism,until kings become philosophers or philosopher...
4,4,alt.atheism,in article bob mcgwier writes however i hate e...


In [15]:
os.makedirs("../data/processed", exist_ok=True)

output_path = "../data/processed/newsgroups_cleaned.csv"

df_final.to_csv(output_path, index=False)

print("Clean dataset saved to:", output_path)

Clean dataset saved to: ../data/processed/newsgroups_cleaned.csv


In [16]:
print(df_final["clean_text"][0][:400])

archive name atheism resources alt atheism archive name resources last modified december version atheist resources addresses of atheist organizations usa freedom from religion foundation darwin fish bumper stickers and assorted other atheist paraphernalia are available from the freedom from religion foundation in the us write to ffrf p o box madison wi telephone evolution designs evolution designs


In [17]:
df_final["clean_text"].str.contains("^>", regex=True).sum()

np.int64(0)

In [18]:
df_final["category"].value_counts()

category
talk.politics.guns          1000
talk.politics.mideast       1000
talk.politics.misc          1000
talk.religion.misc          1000
alt.atheism                  999
rec.sport.baseball           999
rec.sport.hockey             999
sci.crypt                    999
sci.electronics              999
sci.space                    999
comp.windows.x               998
rec.autos                    998
sci.med                      998
comp.graphics                997
comp.sys.ibm.pc.hardware     997
rec.motorcycles              997
soc.religion.christian       997
comp.sys.mac.hardware        996
comp.os.ms-windows.misc      992
misc.forsale                 991
Name: count, dtype: int64